# Predicting the Gender Gap using Machine Learning!

TODO: add notebook description

In [ ]:
# optional: silence warnings for readability
import warnings

warnings.filterwarnings("ignore")

## Data 

### Import libraries

In [ ]:
# data
import pandas as pd
import numpy as np

# plots
import matplotlib.pyplot as plt
import seaborn as sns

### Data import

In [ ]:
dataset = pd.read_csv("data/multipleChoiceResponses.csv")
dataset.head()

### Data ...
TODO: title

In [ ]:
# create a new var with the questions names
question_names = dataset.iloc[0]
question_names

In [ ]:
# dropping questions names flor clarity
dataset = dataset.drop(0, axis=0)
dataset.head()

### Data selection

In [ ]:
# print questions
question_names.head(10)

In [ ]:
# select data of interest
df_short = dataset[["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9"]]
df_short = df_short.rename(
    columns={
        "Q1": "Gender",
        "Q2": "Age",
        "Q3": "Country",
        "Q4": "Education",
        "Q5": "FieldOfStudies",
        "Q6": "JobTitle",
        "Q7": "Industry",
        "Q8": "Experience",
        "Q9": "YearlyCompensation",
    }
)
df_short

In [ ]:
# print info
df_short.info

### Data exploration

### Treat missing values

In [ ]:
# fill NAs with a string (keeping rows, however not providing a lot of info)
df_short = df_short.fillna("Unknown")

In [ ]:
# observe the target values
df_short["YearlyCompensation"].value_counts()

In [ ]:
# drop the rows where the target is not available
df_short = df_short[
    (
        df_short["YearlyCompensation"]
        != "I do not wish to disclose my approximate yearly compensation"
    )
    & (df_short["YearlyCompensation"] != "Unknown")
]
df_short.head()

In [ ]:
# observe the target values
df_short["YearlyCompensation"].value_counts()

### Data types

In [ ]:
# check data types
df_short.info()

#### Features

In [ ]:
# Midpoint mapping for Experience
dic_exp = {
    "0-1": 0.5,
    "1-2": 1.5,
    "2-3": 2.5,
    "3-4": 3.5,
    "4-5": 4.5,
    "5-10": 7.5,
    "10-15": 12.5,
    "15-20": 17.5,
    "20-25": 22.5,
    "25-30": 27.5,
    "30 +": 30,
    "Unknown": 0,
}

# Apply mapping
df_short["Experience"] = df_short["Experience"].map(dic_exp)

In [ ]:
# plot num features distribution
sns.histplot(data=df_short, x="Experience");

In [ ]:
# Midpoint mapping for Age
dic_age = {
    "30-34": 32,
    "22-24": 23,
    "35-39": 37,
    "18-21": 19.5,
    "40-44": 42,
    "25-29": 27,
    "55-59": 57,
    "60-69": 64.5,
    "45-49": 47,
    "50-54": 52,
    "70-79": 74.5,
    "80+": 80,
}

# Apply mapping
df_short["Age"] = df_short["Age"].map(dic_age)

In [ ]:
# plot num features distribution
sns.histplot(data=df_short, x="Age");

In [ ]:
# check data types
df_short.info()

In [ ]:
# categorise features according to datatype
num_vars = ["Age", "Experience"]
cat_vars = [
    "Gender",
    "Country",
    "Education",
    "FieldOfStudies",
    "JobTitle",
    "Industry",
]

## Machine Learning

In [ ]:
from sklearn.model_selection import train_test_split

### Data preparation

#### Split features and target

In [ ]:
# define labels and features
labels_col = "YearlyCompensation"

X = df_short.drop(labels_col, axis=1)
y = df_short[labels_col]

In [ ]:
# display head of features X
X.head()

In [ ]:
# display head of target y
y.head()

#### Encode variables

In [ ]:
# replace objects by numerical categories
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_vars] = enc.fit_transform(X[cat_vars])
X.head()

### The holdout method

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/machine-learning/train_test_split_basic.png" width="400">

In [ ]:
# split into Train/Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

### Fitting a ML model

In [ ]:
# import a model
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression()

In [ ]:
# fit the model
log_model.fit(X_train, y_train)

### Scoring the model performance

In [ ]:
# score the model on the Test data
log_model.score(X_test, y_test)

### Cross validation

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/lectures/machine-learning/K_fold_3.png" width="400">

In [ ]:
from sklearn.model_selection import cross_validate

# 5-Fold Cross validate model
cv_results = cross_validate(log_model, X, y, cv=5)

# scores
print(cv_results["test_score"])

In [ ]:
# mean of scores
scores_mean = float(cv_results["test_score"].mean())
print(f"CV scores mean using a logistic regression model is {scores_mean:.2f}")

**Cross-validation does not output a trained model, it only scores a hypothetical model trained on the entire dataset.**

## Iterate and improve

### Scale the numerical features

In [ ]:
# import a scaler
from sklearn.preprocessing import StandardScaler

# create a reusable variable
std_scaler = StandardScaler()

# create a X copy for safety
X_scaled = X.copy()

# fit the scaler
X_scaled[num_vars] = std_scaler.fit_transform(X[num_vars])

# split into Train/Test
X_train_scaled, X_test, y_train_scaled, y_test = train_test_split(
    X_scaled, y, test_size=0.3
)

In [ ]:
# 5-Fold Cross validate model
cv_results_scaled = cross_validate(log_model, X_scaled, y, cv=5)

# mean of scores
scores_mean_scaled = float(cv_results_scaled["test_score"].mean())
print(
    f"CV scores mean using a scaled logistic regression model is {scores_mean_scaled:.2f}"
)

## Try a better model

In [ ]:
from lightgbm import LGBMClassifier
from lightgbm import plot_importance

In [ ]:
# create the model instance
lgbm_model = LGBMClassifier(verbosity=-1)  # adjusting verbosity to reduce the output

# fit the model
lgbm_model.fit(X_train_scaled, y_train)

# 5-Fold Cross validate model
cv_results_lgbm = cross_validate(lgbm_model, X_scaled, y, cv=5)

# mean of scores
scores_mean_lgbm = float(cv_results_lgbm["test_score"].mean())
print(f"CV scores mean using a scaled LGBM model is {scores_mean_lgbm:.2f}")

## Features importance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_importance(lgbm_model, max_num_features=50, height=0.8, ax=ax)
ax.grid(False)
plt.ylabel("Feature", size=12)
plt.xlabel("Importance", size=12)
plt.title("Features importance with LGBM model", fontsize=15)
plt.show();

In [ ]:
df_short.Country.unique()

## Predict new data

In [ ]:
# create a tech worker profile
Veronica = pd.DataFrame(
    {
        "Gender": ["Female"],
        "Age": [35],
        "Country": ["Italy"],
        "Education": ["Bachelor’s degree"],
        "FieldOfStudies": ["Computer science (software engineering, etc.)"],
        "JobTitle": ["Software Engineer"],
        "Industry": ["Computers/Technology"],
        "Experience": [10],
    }
)

# reuse our encoder to transform the data
Veronica[cat_vars] = enc.transform(Veronica[cat_vars])

# use our fitted model to estimate their yearly pay
Ahsvi_salary = lgbm_model.predict(Veronica)

# display result
print(f"Ahsvi's salary is estimated to be within the range of {Ahsvi_salary[0]} USD")

In [ ]:
# create a tech worker profile
John = pd.DataFrame(
    {
        "Gender": ["Male"],
        "Age": [52],
        "Country": ["Finland"],
        "Education": ["Master’s degree"],
        "FieldOfStudies": ["Computer science (software engineering, etc.)"],
        "JobTitle": ["Software Engineer"],
        "Industry": ["Computers/Technology"],
        "Experience": [20],
    }
)

# reuse our encoder to transform the data
John[cat_vars] = enc.transform(John[cat_vars])

# use our fitted model to estimate their yearly pay
John_salary = lgbm_model.predict(John)

# display result
print(f"John's salary is estimated to be within the range of {John_salary[0]} USD")